# Panneaux de signalisation en conditions dégradées : ResNet sur Colab (T4)

Notebook à exécuter de bout en bout sur un **serveur Colab GPU** depuis VS Code :
**Select Kernel → Colab → New Colab Server → GPU → T4**.

Tout est téléchargé sur le serveur : code (git clone), dépendances, GTSRB, poids ImageNet. Google Drive n'est pas utilisé.
Après une coupure, relancez la cellule 2 puis la cellule d'entraînement : `--resume` repart du dernier checkpoint.

## 1. Vérification du GPU

In [ ]:
!nvidia-smi
import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        "Aucun GPU CUDA détecté. Dans VS Code : Select Kernel -> Colab -> New Colab Server -> GPU -> T4, "
        "puis relancez cette cellule."
    )
print(f"torch {torch.__version__} | GPU : {torch.cuda.get_device_name(0)}")

## 2. Récupération du code et installation

`REPO_URL` pointe vers le dépôt public ; remplacez-le si vous travaillez sur un fork.

In [ ]:
REPO_URL = "https://github.com/saidaghrod1-ops/traffic-sign-robust-resnet.git"  # à changer si vous forkez

import os
if "<votre-utilisateur>" in REPO_URL:
    raise ValueError("Remplacez REPO_URL par l'URL de votre dépôt GitHub.")
os.environ["PYTHONUNBUFFERED"] = "1"  # logs affichés en direct pour les commandes !python
BASE_DIR = "/content" if os.path.isdir("/content") else os.getcwd()
REPO_DIR = os.path.join(BASE_DIR, "traffic-sign-robust-resnet")
# Dépôt privé : exécutez d'abord os.environ["GITHUB_TOKEN"] = getpass.getpass() (jamais de token en dur).
clone_url = REPO_URL
if os.environ.get("GITHUB_TOKEN"):
    clone_url = REPO_URL.replace("https://", f"https://{os.environ['GITHUB_TOKEN']}@")

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git -C "$REPO_DIR" pull --ff-only
else:
    !git clone --depth 1 "$clone_url" "$REPO_DIR"
%cd $REPO_DIR
!pip install -q -r requirements.txt

## 3. Données GTSRB et grille des dégradations

Télécharge GTSRB (~300 Mo), affiche le split par track et met le jeu de test en cache (tenseur 64×64).

In [ ]:
!python -m src.data --examples
!python -m src.corruptions --grid
from IPython.display import Image, Markdown, display
display(Image("results/gtsrb_examples.png"))
display(Image("results/corruption_grid.png"))

## 4. Entraînement de la baseline

Images propres, 15 époques. `last.pt` est écrit à chaque époque : relancer la cellule reprend l'entraînement (ou s'arrête aussitôt s'il est terminé).

Optionnel : pour survivre au **recyclage de la VM**, décommentez les lignes du haut. Les checkpoints seront copiés sur un dépôt privé du Hugging Face Hub, puis récupérés par `--resume`.

In [ ]:
# import os, getpass
# os.environ["HF_REPO_ID"] = "<utilisateur-hf>/traffic-sign-robust-resnet"
# os.environ["HF_TOKEN"] = os.environ.get("HF_TOKEN") or getpass.getpass("Token Hugging Face (write) : ")

!python -m src.train --config configs/baseline.yaml --resume
display(Image("results/baseline/curves.png"))

## 5. Entraînement du modèle robuste

Avec une probabilité de 0,5, chaque batch reçoit une dégradation aléatoire (type et niveau). `snow` et `motion_blur` restent hors entraînement.

In [ ]:
!python -m src.train --config configs/robust.yaml --resume
display(Image("results/robust/curves.png"))

## 6. Évaluation complète

Précision propre et pour 9 types × 5 niveaux, CE / mCE (baseline = 100 %), ECE (15 bins), matrices de confusion. La section Résultats du README est mise à jour sur le serveur.

In [ ]:
!python -m src.evaluate --runs baseline robust --baseline baseline --update-readme
display(Markdown(open("results/robustness_table.md", encoding="utf-8").read()))
for name in ["acc_vs_severity", "heatmap_baseline", "heatmap_robust", "training_curves",
             "confusion_baseline_clean", "confusion_robust_clean", "confusion_baseline_s5", "confusion_robust_s5"]:
    display(Image(f"results/figures/{name}.png"))

## 7. Adaptation BatchNorm au test

Statistiques BN réestimées sur 2 048 images dégradées non étiquetées, sans gradient : comparaison sans / avec adaptation.

In [ ]:
!python -m src.bn_adapt --runs baseline robust --update-readme
display(Markdown(open("results/bn_adapt_table.md", encoding="utf-8").read()))
display(Image("results/figures/bn_adapt.png"))

## 8. Grad-CAM

Cartes sur `layer4` : image propre vs dégradée, baseline vs robust.

In [ ]:
!python -m src.gradcam --runs baseline robust --corruptions fog snow --severity 4
display(Image("results/figures/gradcam_fog_s4.png"))
display(Image("results/figures/gradcam_snow_s4.png"))

## 9. Archive des résultats

`results.zip` contient figures, tableaux, logs, `best.pt` et le README mis à jour. Les `last.pt` sont exclus. Le fichier reste sur le serveur Colab (`google.colab.files.download` ne fonctionne pas dans VS Code) : utilisez la cellule suivante pour l'envoyer sur le Hugging Face Hub.

In [ ]:
import zipfile
from pathlib import Path

archive = Path("results.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(Path("results").rglob("*")):
        if f.is_file() and (f.suffix != ".pt" or f.name == "best.pt"):
            zf.write(f)
    zf.write("README.md")
print(f"{archive.resolve()} : {archive.stat().st_size / 1e6:.1f} Mo")

### 9 bis. (Optionnel) Upload vers le Hugging Face Hub

Le token est lu depuis la variable d'environnement `HF_TOKEN`. S'il est absent, il est demandé par `getpass`, jamais écrit dans le notebook.

In [ ]:
import getpass
import os
from huggingface_hub import HfApi

if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass.getpass("Token Hugging Face (write) : ")
HF_REPO_ID = os.environ.get("HF_REPO_ID", "<utilisateur-hf>/traffic-sign-robust-resnet")  # <- à remplacer
if HF_REPO_ID.startswith("<"):
    raise ValueError("Définissez HF_REPO_ID (ex. os.environ['HF_REPO_ID'] = 'moi/traffic-sign-robust-resnet').")

api = HfApi(token=os.environ["HF_TOKEN"])
api.create_repo(HF_REPO_ID, private=True, exist_ok=True)
api.upload_file(path_or_fileobj="results.zip", path_in_repo="results.zip", repo_id=HF_REPO_ID)
print(f"https://huggingface.co/{HF_REPO_ID}/blob/main/results.zip")